In [10]:
# CELL 1 - Load base dataset and sort for per-customer time series
import pandas as pd
import numpy as np

df = pd.read_csv("../data/processed/base_transactions.csv")
df["TransactionDT"] = pd.to_datetime(df["TransactionDT"])

# Sort by customer THEN time: required so groupby-rolling output
# row order matches df row order (we assign positionally below).
df = df.sort_values(["customer_id", "TransactionDT"]).reset_index(drop=True)
n_rows_in = len(df)
print(f"Loaded: {df.shape}")

Loaded: (590540, 56)


In [11]:
# CELL 2 - Datetime index + shifted "prior amount" (leakage fix)
df = df.set_index("TransactionDT")

# Shift each customer's amounts by one row. Rolling windows computed on
# amt_prior describe PRIOR behavior only - a transaction is never part
# of its own baseline, so deviation features are not diluted toward zero.
df["amt_prior"] = df.groupby("customer_id")["TransactionAmt"].shift(1)

In [12]:
# CELL 3 - Rolling velocity + amount features (one pass per window)
def add_window_features(df, window, suffix):
    """Per-customer rolling stats over PRIOR transactions for one window."""
    rolled = (
        df.groupby("customer_id")["amt_prior"]
          .rolling(window)
          .agg(["count", "mean", "max", "std"])   # single pass, 4 stats
          .reset_index(level=0, drop=True)
    )
    df[f"txn_count_{suffix}"] = rolled["count"].to_numpy()
    df[f"avg_amt_{suffix}"]   = rolled["mean"].to_numpy()
    df[f"max_amt_{suffix}"]   = rolled["max"].to_numpy()
    df[f"std_amt_{suffix}"]   = rolled["std"].to_numpy()
    return df

df = add_window_features(df, "1h",  "1h")    # velocity: last hour
df = add_window_features(df, "24h", "24h")   # velocity: last day
df = add_window_features(df, "7D",  "7d")    # velocity: last week

In [13]:
# CELL 4 - Baseline deviation: current transaction vs PRIOR baseline
df["amount_dev_24h"] = df["TransactionAmt"] - df["avg_amt_24h"]

# Guard: std == 0 (flat baseline) or std NaN (< 2 priors) would give
# inf/NaN. amount_dev_24h still captures the raw deviation in those cases.
df["amount_zscore_24h"] = np.where(
    df["std_amt_24h"] > 0,
    df["amount_dev_24h"] / df["std_amt_24h"],
    0,
)

In [14]:
# CELL 5 - History flags: convert "no prior data" into a signal
# (must run BEFORE fillna, while NaN still means "no history")
df["no_prior_1h"]      = df["txn_count_1h"].isna().astype(int)
df["no_prior_24h"]     = df["txn_count_24h"].isna().astype(int)   # first-ever txn
df["single_prior_24h"] = (df["txn_count_24h"] == 1).astype(int)   # std undefined

In [15]:
# CELL 6 - Time-of-day risk features
# (1970 dates are fine: day boundaries align every 86,400 seconds,
#  so hour-of-day is preserved relative to the true reference)
df["hour"]         = df.index.hour
df["day_of_week"]  = df.index.dayofweek
df["is_night_txn"] = df["hour"].isin([0, 1, 2, 3, 4]).astype(int)

In [16]:
# CELL 7 - Targeted missing-value handling (replaces global dropna)
count_cols = ["txn_count_1h", "txn_count_24h", "txn_count_7d"]
df[count_cols] = df[count_cols].fillna(0)      # NaN count = zero prior txns

stat_cols = [
    "avg_amt_1h", "avg_amt_24h", "avg_amt_7d",
    "max_amt_1h", "max_amt_24h", "max_amt_7d",
    "std_amt_1h", "std_amt_24h", "std_amt_7d",
    "amount_dev_24h", "amount_zscore_24h",
]
df[stat_cols] = df[stat_cols].fillna(0)        # neutral value; flags carry info

# Headline check: no rows were deleted. First / one-shot transactions
# (a high-fraud-risk segment) stay in the dataset.
assert len(df) == n_rows_in, f"Row count changed: {n_rows_in} -> {len(df)}"
print(f"Rows preserved: {len(df)} / {n_rows_in}")

Rows preserved: 590540 / 590540


In [17]:
# CELL 8 - Sanity check: do the features separate fraud from legit?
check_cols = [
    "txn_count_1h", "txn_count_24h", "amount_zscore_24h",
    "is_night_txn", "no_prior_24h",
]
print(df.groupby("isFraud")[check_cols].mean().round(3))
print()
print(df[["txn_count_1h", "txn_count_24h", "avg_amt_24h",
          "amount_zscore_24h"]].describe().round(3))

         txn_count_1h  txn_count_24h  amount_zscore_24h  is_night_txn  \
isFraud                                                                 
0               2.426         19.738              0.886         0.225   
1               2.281         15.921              0.831         0.231   

         no_prior_24h  
isFraud                
0                 0.0  
1                 0.0  

       txn_count_1h  txn_count_24h  avg_amt_24h  amount_zscore_24h
count    590540.000     590540.000   590540.000         590540.000
mean          2.420         19.605      130.281              0.884
std           5.505         45.572      147.685             78.106
min           0.000          0.000        0.000          -1724.633
25%           1.000          2.000       58.950             -0.562
50%           1.000          6.000       99.971              0.000
75%           2.000         20.000      149.954              0.127
max         192.000        881.000     4879.950          49847.268


In [18]:
# CELL 9 - Save feature dataset
df_feat = df.drop(columns=["amt_prior"]).reset_index()
df_feat.to_csv("../data/processed/time_window_features.csv", index=False)
print(f"Saved: {df_feat.shape}")

Saved: (590540, 76)
